In [1]:
import requests
import json
import pandas as pd
import os
import time
import numpy as np

os.chdir(r"C:\Users\Mohan2\OneDrive - University of Ottawa\THESIS\test_set")

In [2]:
#get list of the taxid's for all entries in SCOPe

lines = []
with open("dir.des.scope.2.08-stable.txt","r") as f:
    for line_no, line in enumerate(f):
        if line.split()[1]=="sp":
            lines.append(str(line_no) + " " + line)
            
n_line = [x.split()[0] for x in lines]
fold = [x.split()[3] for x in lines]
Spec = [" ".join(x.split()[5:]) for x in lines]
# " ".join(lines[0].split()[4:])
df = pd.DataFrame({"fold": fold, "spec": Spec}, index= n_line)

drop = df[~df['spec'].str.contains('TaxId')].index
df.drop(drop, inplace=True)
df

#only keep the TaxIDs
df['taxid'] = df['spec'].str.split("TaxId:",expand=True)[1].str[:-1]
df['taxid'] = df['taxid'].str.strip()
df.drop(axis=1, labels=['spec'], inplace=True)
df

,fold,taxid
9,a.1.1.1,1423
11,a.1.1.1,5885
14,a.1.1.1,3054
17,a.1.1.1,1773
34,a.1.1.1,1773
...,...,...
362200,k.13.1.1,1280
362462,k.37.1.1,32630
362467,k.37.1.1,32644
362475,k.37.1.1,32630


In [3]:
#get the full name for each taxid
from Bio import Entrez
Entrez.email = 'mrake012@uottawa.ca'
handle = Entrez.efetch(db="Taxonomy", id=','.join(df['taxid'].unique()), retmode="xml") #From biopython tutorial
records = Entrez.read(handle) 
handle.close()

In [4]:
AKA_IDs = {int(x['AkaTaxIds'][0]) : int(x['TaxId']) for x in records if 'AkaTaxIds' in x.keys()}
record_dict = {int(x['TaxId']):' '.join(x['Lineage'].split(';')) for x in records}
record_dict = {**record_dict, **{x:record_dict[AKA_IDs[x]] for x in AKA_IDs.keys()}}

In [8]:
df = df.astype({'taxid': 'int32'})
taxonomy = []
for x in df['taxid']:
    try:
        if x in AKA_IDs.keys():
            taxonomy.append(record_dict[AKA_IDs[x]])
        else:
            taxonomy.append(record_dict[x])
    except Exception as e:
        taxonomy.append('X')
        print(x, e)

20894 20894


In [9]:
df['taxonomy'] = taxonomy
df.to_pickle("SCOP_taxonomy.pkl")
df

,fold,taxid,taxonomy
9,a.1.1.1,1423,cellular organisms Bacteria Terrabacteria gr...
11,a.1.1.1,5885,cellular organisms Eukaryota Sar Alveolata ...
14,a.1.1.1,3054,cellular organisms Eukaryota Viridiplantae ...
17,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
34,a.1.1.1,1773,cellular organisms Bacteria Terrabacteria gr...
...,...,...,...
362200,k.13.1.1,1280,cellular organisms Bacteria Terrabacteria gr...
362462,k.37.1.1,32630,other entries other sequences artificial seq...
362467,k.37.1.1,32644,unclassified entries unclassified sequences
362475,k.37.1.1,32630,other entries other sequences artificial seq...
